# ATTR coding explorer + confirmed finder

## Workflow (run in order)

1. **Step A - Broad amyloidosis explore**  
   **Per column:** for each column, COUNT → if hits, show rows + exact values.  
   Easy to see which column holds amyloid/ATTR text (including odd fields due to data trasformation not as expected in tables -ex: Firm_golbal_id include text data as well).

2. **Step B - Confirmed ATTR**  
   Same **per-column** pattern with ICD/SNOMED + ATTR NLP (codes also as text NLP reason: lab result include icd code as well in a note).

Each table section shows matching rows, then **exact column match details** (full cell value, not a snippet).

**Storage:** reads only + optional session `TEMPORARY` tables. No permanent warehouse create/update.


## 1 - Session

In [ ]:
import pandas as pd
import re
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE DATABASE QUALDERM_MODMED_NEW').collect()
session.sql('USE SCHEMA PUBLIC').collect()
print(session.get_current_database(), session.get_current_schema())
print('Total PATIENT rows:', session.sql('SELECT COUNT(*) FROM PATIENT').collect()[0][0])

## 2 — Dictionaries + helpers

In [ ]:
# ----- Step A: broad amyloidosis / amyloid variants (explore only) -----
BROAD_AMYLOID_NLP = [
    '%amyloidosis%',
    '%amyloidoses%',
    '%amyloidose%',          # spelling variant
    '%amiloidosis%',         # common misspelling
    '%amiloid%',
    '%amyloid%',             # catches amyloid cardiomyopathy, etc.
    '%amyloid %',
    '% amyloid%',
    '%amyloid-%',
    '%cardiac amyloid%',
    '%amyloid cardiomyopathy%',
    '%amyloid heart%',
    '%primary amyloid%',
    '%secondary amyloid%',
    '%familial amyloid%',
    '%senile amyloid%',
    '%systemic amyloid%',
    '%hereditary amyloid%',
    # ICD / SNOMED as text (codes often sit inside lab/path notes)
    '%E85.82%', '%E8582%',
    '%E85.1%', '%E851%',
    '%E85.81%', '%E8581%',
    '%E85.89%', '%E8589%',
    '%E85.%', '%E85 %', '% E85%',  # broader E85 family in free text
    '%237877004%', '%16573007%', '%42295001%', '%442012008%',
]

# ----- Step B: confirmed ATTR -----
ATTR_SNOMED = [
    '237877004',
    '16573007',
    '42295001',
    '442012008',
    '715655000'
]
ATTR_ICD = ['E85.82', 'E85.1']

# Word / phrase NLP
ATTR_NLP_WORDS = [
    '%transthyretin%', '%transthyretin amyloidosis%',
    '%amyloidogenic transthyretin%',
    '%ttr amyloidosis%', '%ttr amyloid%',
    '%attr amyloidosis%', '%attr amyloid%',
    '%wild-type attr%', '%wild type attr%',
    '%attrwt%', '%attr-wt%', '%wtattr%', '%wt-attr%',
    '%attrv%', '%attr-v%', '%vattr%', '%v-attr%',
    '%attr-cm%', '%attr cm%', '%attrcm%',
    '%hereditary attr%', '%hatttr%', '%h-attr%',
    '%familial amyloid polyneuropathy%',
    '%senile systemic amyloidosis%', '%senile cardiac amyloidosis%',
    '%ttr mutation%', '%ttr gene%',
    '%val122ile%', '%v122i%', '%thr60ala%', '%t60a%',
    '%tafamidis%', '%vyndaqel%', '%vyndamax%',
    '%patisiran%', '%onpattro%', '%amvuttra%', '%vutrisiran%',
    '%inotersen%', '%tegsedi%',
]

# ICD + SNOMED as NLP so codes inside LAB_RESULT / notes are not missed
ATTR_CODE_NLP = []
for _code in ATTR_ICD:
    ATTR_CODE_NLP.append(f'%{_code}%')
    ATTR_CODE_NLP.append(f'%{_code.replace(".", "")}%')
for _sid in ATTR_SNOMED:
    ATTR_CODE_NLP.append(f'%{_sid}%')

# Combined search list used in Step B (words + codes as text)
ATTR_NLP = ATTR_NLP_WORDS + ATTR_CODE_NLP

SNOMED_IN = ', '.join([f"'{s}'" for s in ATTR_SNOMED])
META_COLS = {'MATCH_TYPE', 'PHENOTYPE_HINT', 'SRC_TABLE'}

EXPLORE_TABLES = [
    'VISIT_CODE',
    'MEDICAL_HISTORY',
    'SURGICAL_HISTORY',
    'FAMILY_HISTORY',
    'SPECIALTY_FAMILY_HISTORY',
    'SPECIALTY_HISTORY',
    'SOCIAL_HISTORY',
    'LAB_RESULT',
    'VISIT',
]

# Fast path for huge VISIT_CODE — do NOT NLP-scan every column
VISIT_CODE_SEARCH_COLS = ['CODE_VALUE', 'CODE_SYSTEM', 'SOURCE']


def intersect_cols(available, wanted):
    av = {str(c).upper(): c for c in available}
    out = []
    for w in wanted:
        if w.upper() in av:
            out.append(av[w.upper()])
    if not out:
        # fallback: CODE_VALUE only if present
        if 'CODE_VALUE' in av:
            out = [av['CODE_VALUE']]
    return out


def snomed_pred(*cols):
    return '(' + ' OR '.join([f"TO_VARCHAR({c}) IN ({SNOMED_IN})" for c in cols]) + ')'


def icd_pred(col):
    parts = []
    for code in ATTR_ICD:
        bare = code.replace('.', '')
        parts.append(
            f"STARTSWITH(UPPER(REPLACE(COALESCE(TO_VARCHAR({col}), ''), '.', '')), '{bare}')"
        )
        parts.append(f"STARTSWITH(UPPER(COALESCE(TO_VARCHAR({col}), '')), '{code}')")
    return '(' + ' OR '.join(parts) + ')'


def nlp_pred(*cols, term_list=None, include_attr_regex=False):
    terms = term_list if term_list is not None else ATTR_NLP
    parts = []
    for c in cols:
        for p in terms:
            parts.append(f"UPPER(COALESCE(TO_VARCHAR({c}), '')) LIKE UPPER('{p}')")
        if include_attr_regex:
            parts.append(
                f"REGEXP_LIKE(UPPER(COALESCE(TO_VARCHAR({c}), '')), '(^|[^A-Z0-9])ATTR([^A-Z0-9]|$)|(^|[^A-Z0-9])ATTR-')"
            )
    return '(' + ' OR '.join(parts) + ')'


def nlp_all_columns_sql(alias, columns, term_list, include_attr_regex=False):
    exprs = [f'TO_VARCHAR({alias}.{c})' for c in columns]
    return nlp_pred(
        *exprs, term_list=term_list, include_attr_regex=include_attr_regex
    )


def nlp_named_columns_sql(alias, columns, term_list, include_attr_regex=False):
    """NLP only on an allowlist of columns (VISIT_CODE fast path)."""
    if not columns:
        return '(FALSE)'
    return nlp_all_columns_sql(
        alias, columns, term_list, include_attr_regex=include_attr_regex
    )


def phenotype_label_sql(code_col):
    return f"""
    CASE
      WHEN TO_VARCHAR({code_col}) IN ('237877004')
        OR STARTSWITH(UPPER(REPLACE(COALESCE(TO_VARCHAR({code_col}),''),'.','')), 'E8582')
        THEN 'ATTRwt / E85.82'
      WHEN TO_VARCHAR({code_col}) IN ('16573007')
        THEN 'ATTR-CM senile cardiac (SNOMED 16573007)'
      WHEN TO_VARCHAR({code_col}) IN ('42295001')
        OR STARTSWITH(UPPER(REPLACE(COALESCE(TO_VARCHAR({code_col}),''),'.','')), 'E851')
        THEN 'hATTR-PN / FAP / E85.1'
      WHEN TO_VARCHAR({code_col}) IN ('442012008')
        THEN 'Broad amyloidogenic TTR (SNOMED 442012008)'
      WHEN STARTSWITH(UPPER(REPLACE(COALESCE(TO_VARCHAR({code_col}),''),'.','')), 'E8581')
        THEN 'E85.81'
      WHEN STARTSWITH(UPPER(REPLACE(COALESCE(TO_VARCHAR({code_col}),''),'.','')), 'E8589')
        THEN 'E85.89'
      ELSE 'NLP / other match'
    END
    """


def table_columns(table_name):
    df = session.sql(f"""
        SELECT COLUMN_NAME
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_CATALOG = CURRENT_DATABASE()
          AND TABLE_SCHEMA = CURRENT_SCHEMA()
          AND TABLE_NAME = '{table_name.upper()}'
        ORDER BY ORDINAL_POSITION
    """).to_pandas()
    cols = [str(c) for c in df['COLUMN_NAME'].tolist()]
    print(f'{table_name} columns ({len(cols)}):', cols)
    return cols


def _pid_col(df):
    if df is None or len(df) == 0:
        return None
    if 'PATIENT_ID' in df.columns:
        return 'PATIENT_ID'
    for c in df.columns:
        if str(c).upper() == 'PATIENT_ID':
            return c
    return None


def store_table(name, df):
    """In-memory only. No warehouse write."""
    if len(df):
        pid = _pid_col(df)
        if pid is not None:
            df[pid] = df[pid].astype(str).str.strip()
            ids = set(df[pid])
        else:
            ids = set()
    else:
        ids = set()
    table_hits[name] = ids
    table_frames[name] = df
    print(f'{name} match rows: {len(df)} | unique patients: {len(ids)}')
    display(df)
    return df


def show_nlp_column_details(
    table_name, df, term_list, include_attr_regex=False, scan_cols_only=None
):
    """Exact full column values that caused each NLP hit + full row context."""
    print(f'\n===== {table_name}: NLP hits with EXACT column values =====')
    if df is None or len(df) == 0:
        print('(no matching rows)')
        return None, None, None

    pid = _pid_col(df)
    if scan_cols_only:
        want = {str(c).upper() for c in scan_cols_only}
        scan_cols = [
            c for c in df.columns
            if str(c).upper() in want and str(c).upper() not in META_COLS
        ]
    else:
        scan_cols = [c for c in df.columns if str(c).upper() not in META_COLS]
    print('Scanning columns for NLP detail:', [str(c) for c in scan_cols])
    detail_rows = []

    for _, row in df.iterrows():
        patient = str(row[pid]) if pid is not None else None
        for col in scan_cols:
            val = row[col]
            if val is None:
                continue
            try:
                if pd.isna(val):
                    continue
            except (TypeError, ValueError):
                pass
            exact = str(val)
            if exact.strip() == '' or exact.lower() in ('none', 'nan'):
                continue
            exact_u = exact.upper()
            hit_terms = []
            for term in term_list:
                needle = term.strip('%').upper()
                if needle and needle in exact_u:
                    hit_terms.append(term)
            if include_attr_regex and re.search(
                r'(^|[^A-Z0-9])ATTR([^A-Z0-9]|$)|(^|[^A-Z0-9])ATTR-', exact_u
            ):
                hit_terms.append('ATTR-TOKEN(regex)')

            for term in hit_terms:
                rec = {
                    'PATIENT_ID': patient,
                    'NLP_TERM': term,
                    'MATCH_COLUMN': col,
                    'EXACT_COLUMN_VALUE': exact,
                }
                for c2 in df.columns:
                    rec[f'ROW_{c2}'] = row[c2]
                detail_rows.append(rec)

    df_detail = pd.DataFrame(detail_rows)
    if len(df_detail) == 0:
        print('(no NLP term found in scanned columns)')
        display(df_detail)
        return df_detail, None, None

    display(df_detail)

    print(f'\n===== {table_name}: keyword counts =====')
    df_term = (
        df_detail.groupby('NLP_TERM', as_index=False)
        .agg(
            HIT_ROWS=('EXACT_COLUMN_VALUE', 'size'),
            UNIQUE_PATIENTS=('PATIENT_ID', 'nunique'),
            COLUMNS_HIT=('MATCH_COLUMN', lambda s: ', '.join(sorted(set(map(str, s))))),
        )
        .sort_values('UNIQUE_PATIENTS', ascending=False)
    )
    display(df_term)

    print(f'\n===== {table_name}: which COLUMNS held the text =====')
    df_col = (
        df_detail.groupby('MATCH_COLUMN', as_index=False)
        .agg(
            HIT_ROWS=('EXACT_COLUMN_VALUE', 'size'),
            UNIQUE_PATIENTS=('PATIENT_ID', 'nunique'),
            TERMS=('NLP_TERM', lambda s: ', '.join(sorted(set(map(str, s)))[:15])),
        )
        .sort_values('HIT_ROWS', ascending=False)
    )
    display(df_col)
    return df_detail, df_term, df_col


def show_code_breakdown(table_name, df, code_col='CODE_VALUE'):
    print(f'\n===== {table_name}: CODE_VALUE counts =====')
    if df is None or len(df) == 0:
        print('(no rows)')
        return None, None
    pid = _pid_col(df)
    colmap = {str(c).upper(): c for c in df.columns}
    if code_col.upper() not in colmap:
        return None, None
    ccol = colmap[code_col.upper()]
    tmp = df.copy()
    tmp['_CODE'] = tmp[ccol].astype(str)
    g = (
        tmp.groupby('_CODE', as_index=False)
        .agg(MATCH_ROWS=('_CODE', 'size'), UNIQUE_PATIENTS=(pid, 'nunique'))
        .rename(columns={'_CODE': 'CODE_VALUE'})
        .sort_values('UNIQUE_PATIENTS', ascending=False)
    )
    if 'PHENOTYPE_HINT' in df.columns:
        ph = (
            tmp.groupby('_CODE')['PHENOTYPE_HINT']
            .agg(lambda s: ', '.join(sorted(set(map(str, s)))))
            .reset_index()
            .rename(columns={'_CODE': 'CODE_VALUE'})
        )
        g = g.merge(ph, on='CODE_VALUE', how='left')
    display(g)
    return g, None


def run_per_column_scan(
    table,
    term_list,
    key_prefix,
    include_attr_regex=False,
    columns=None,
    extra_where_by_col=None,
):
    """Scan ONE column at a time: COUNT → pull rows → exact NLP detail.

    extra_where_by_col: optional dict {COLUMN_NAME_UPPER: sql_bool_expr}
    OR'd with NLP for that column only (e.g. structured ICD on CODE_VALUE).
    """
    cols = columns if columns is not None else table_columns(table)
    extra_where_by_col = extra_where_by_col or {}
    extra_u = {str(k).upper(): v for k, v in extra_where_by_col.items()}

    col_summaries = []
    detail_parts = []
    row_parts = []
    all_ids = set()

    for col in cols:
        print('\n' + '=' * 64)
        print(f'>>> {table}.{col}')
        print('=' * 64)

        pred_parts = [
            nlp_named_columns_sql(
                'T', [col], term_list, include_attr_regex=include_attr_regex
            )
        ]
        if str(col).upper() in extra_u:
            pred_parts.append(f"({extra_u[str(col).upper()]})")

        where_sql = (
            f"T.PATIENT_ID IS NOT NULL AND ({' OR '.join(pred_parts)})"
        )

        n_sql = session.sql(
            f"SELECT COUNT(*) AS N FROM {table} T WHERE {where_sql}"
        ).collect()[0][0]
        n_pat = session.sql(f"""
            SELECT COUNT(DISTINCT TRIM(T.PATIENT_ID)) AS N
            FROM {table} T
            WHERE {where_sql}
        """).collect()[0][0]
        print(f'SQL count: rows={n_sql} | unique patients={n_pat}')
        col_summaries.append({
            'TABLE': table,
            'COLUMN': str(col),
            'MATCH_ROWS': int(n_sql),
            'UNIQUE_PATIENTS': int(n_pat),
        })

        if int(n_sql) == 0:
            print('(no rows — next column)')
            continue

        df = session.sql(f"""
            SELECT T.*
            FROM {table} T
            WHERE {where_sql}
        """).to_pandas()

        col_key = f'{key_prefix}__{col}'
        store_table(col_key, df)
        all_ids |= table_hits.get(col_key, set())

        df_tag = df.copy()
        df_tag['_SCAN_COLUMN'] = str(col)
        row_parts.append(df_tag)

        dfd = show_nlp_column_details(
            col_key,
            df,
            term_list,
            include_attr_regex=include_attr_regex,
            scan_cols_only=[col],
        )[0]
        if dfd is not None and len(dfd):
            dfd = dfd.copy()
            dfd['SCAN_COLUMN'] = str(col)
            detail_parts.append(dfd)

    # Roll up for this table (used by Step A summary / Step B union)
    table_hits[key_prefix] = all_ids
    table_frames[key_prefix] = (
        pd.concat(row_parts, ignore_index=True, sort=False)
        if row_parts else pd.DataFrame()
    )
    nlp_detail_frames[key_prefix] = (
        pd.concat(detail_parts, ignore_index=True, sort=False)
        if detail_parts else pd.DataFrame()
    )

    df_col_summary = pd.DataFrame(col_summaries).sort_values(
        'UNIQUE_PATIENTS', ascending=False
    )
    print(f'\n===== {table}: per-column summary =====')
    display(df_col_summary)
    print(
        f'{key_prefix} rolled-up unique patients:',
        len(all_ids),
        '| detail rows:',
        len(nlp_detail_frames[key_prefix]),
    )
    return df_col_summary


table_hits = {}
table_frames = {}
nlp_detail_frames = {}
column_scan_summaries = []  # optional collect across tables

print('BROAD terms:', len(BROAD_AMYLOID_NLP))
print('ATTR word NLP:', len(ATTR_NLP_WORDS))
print('ATTR code-as-NLP (ICD/SNOMED):', ATTR_CODE_NLP)
print('ATTR NLP total (words+codes):', len(ATTR_NLP))
print('Explore tables:', EXPLORE_TABLES)
print('Mode: PER-COLUMN scan (COUNT then pull if hits)')

## Step A — Broad amyloidosis explore

**Not confirmed ATTR.** **Per-column** scan to learn:
- which columns hold amyloid-related text (including odd fields)
- which broad keywords fire

Each column: COUNT → if 0 skip → else rows + exact values → table column summary.
Review here, then Step B.

### A.0 Broad term list (review)

In [ ]:
display(pd.DataFrame({'BROAD_AMYLOID_NLP': BROAD_AMYLOID_NLP}))
print('Note: %amyloid% is broad — expects AL + ATTR + unspecified.')

### A.1 `VISIT_CODE` — broad amyloidosis (**per column**)

Runs **one column at a time** with broad keywords/codes:
1. SQL `COUNT` for that column
2. If hits → pull rows + exact column values
3. Column summary at the end of this table

Skip empty columns quickly; stop the cell anytime after you’ve seen enough.

In [ ]:
_ = run_per_column_scan(
    table='VISIT_CODE',
    term_list=BROAD_AMYLOID_NLP,
    key_prefix='BROAD_VISIT_CODE',
    include_attr_regex=False,
)

### A.2 `MEDICAL_HISTORY` — broad amyloidosis (**per column**)

Runs **one column at a time** with broad keywords/codes:
1. SQL `COUNT` for that column
2. If hits → pull rows + exact column values
3. Column summary at the end of this table

Skip empty columns quickly; stop the cell anytime after you’ve seen enough.

In [ ]:
_ = run_per_column_scan(
    table='MEDICAL_HISTORY',
    term_list=BROAD_AMYLOID_NLP,
    key_prefix='BROAD_MEDICAL_HISTORY',
    include_attr_regex=False,
)

### A.3 `SURGICAL_HISTORY` — broad amyloidosis (**per column**)

Runs **one column at a time** with broad keywords/codes:
1. SQL `COUNT` for that column
2. If hits → pull rows + exact column values
3. Column summary at the end of this table

Skip empty columns quickly; stop the cell anytime after you’ve seen enough.

In [ ]:
_ = run_per_column_scan(
    table='SURGICAL_HISTORY',
    term_list=BROAD_AMYLOID_NLP,
    key_prefix='BROAD_SURGICAL_HISTORY',
    include_attr_regex=False,
)

### A.4 `FAMILY_HISTORY` — broad amyloidosis (**per column**)

Runs **one column at a time** with broad keywords/codes:
1. SQL `COUNT` for that column
2. If hits → pull rows + exact column values
3. Column summary at the end of this table

Skip empty columns quickly; stop the cell anytime after you’ve seen enough.

In [ ]:
_ = run_per_column_scan(
    table='FAMILY_HISTORY',
    term_list=BROAD_AMYLOID_NLP,
    key_prefix='BROAD_FAMILY_HISTORY',
    include_attr_regex=False,
)

### A.5 `SPECIALTY_FAMILY_HISTORY` — broad amyloidosis (**per column**)

Runs **one column at a time** with broad keywords/codes:
1. SQL `COUNT` for that column
2. If hits → pull rows + exact column values
3. Column summary at the end of this table

Skip empty columns quickly; stop the cell anytime after you’ve seen enough.

In [ ]:
_ = run_per_column_scan(
    table='SPECIALTY_FAMILY_HISTORY',
    term_list=BROAD_AMYLOID_NLP,
    key_prefix='BROAD_SPECIALTY_FAMILY_HISTORY',
    include_attr_regex=False,
)

### A.6 `SPECIALTY_HISTORY` — broad amyloidosis (**per column**)

Runs **one column at a time** with broad keywords/codes:
1. SQL `COUNT` for that column
2. If hits → pull rows + exact column values
3. Column summary at the end of this table

Skip empty columns quickly; stop the cell anytime after you’ve seen enough.

In [ ]:
_ = run_per_column_scan(
    table='SPECIALTY_HISTORY',
    term_list=BROAD_AMYLOID_NLP,
    key_prefix='BROAD_SPECIALTY_HISTORY',
    include_attr_regex=False,
)

### A.7 `SOCIAL_HISTORY` — broad amyloidosis (**per column**)

Runs **one column at a time** with broad keywords/codes:
1. SQL `COUNT` for that column
2. If hits → pull rows + exact column values
3. Column summary at the end of this table

Skip empty columns quickly; stop the cell anytime after you’ve seen enough.

In [ ]:
_ = run_per_column_scan(
    table='SOCIAL_HISTORY',
    term_list=BROAD_AMYLOID_NLP,
    key_prefix='BROAD_SOCIAL_HISTORY',
    include_attr_regex=False,
)

### A.8 `LAB_RESULT` — broad amyloidosis (**per column**)

Runs **one column at a time** with broad keywords/codes:
1. SQL `COUNT` for that column
2. If hits → pull rows + exact column values
3. Column summary at the end of this table

Skip empty columns quickly; stop the cell anytime after you’ve seen enough.

In [ ]:
_ = run_per_column_scan(
    table='LAB_RESULT',
    term_list=BROAD_AMYLOID_NLP,
    key_prefix='BROAD_LAB_RESULT',
    include_attr_regex=False,
)

### A.9 `VISIT` — broad amyloidosis (**per column**)

Runs **one column at a time** with broad keywords/codes:
1. SQL `COUNT` for that column
2. If hits → pull rows + exact column values
3. Column summary at the end of this table

Skip empty columns quickly; stop the cell anytime after you’ve seen enough.

In [ ]:
_ = run_per_column_scan(
    table='VISIT',
    term_list=BROAD_AMYLOID_NLP,
    key_prefix='BROAD_VISIT',
    include_attr_regex=False,
)

### A.10 Step A cross-table summary

Use this to decide which columns/keywords matter before Step B.

In [ ]:
broad_summary = []
for t in EXPLORE_TABLES:
    key = f'BROAD_{t}'
    n_pat = len(table_hits.get(key, set()))
    n_row = len(table_frames.get(key, pd.DataFrame()))
    dfd = nlp_detail_frames.get(key)
    top_cols = ''
    top_terms = ''
    if dfd is not None and len(dfd):
        top_cols = ', '.join(
            dfd['MATCH_COLUMN'].value_counts().head(5).index.astype(str).tolist()
        )
        top_terms = ', '.join(
            dfd['NLP_TERM'].value_counts().head(5).index.astype(str).tolist()
        )
    broad_summary.append({
        'TABLE': t,
        'MATCH_ROWS': n_row,
        'UNIQUE_PATIENTS': n_pat,
        'TOP_MATCH_COLUMNS': top_cols,
        'TOP_KEYWORDS': top_terms,
    })

df_broad_summary = pd.DataFrame(broad_summary).sort_values(
    'UNIQUE_PATIENTS', ascending=False
)
print('=== Step A: per-table broad amyloidosis footprint ===')
display(df_broad_summary)

# All broad exact-detail rows stacked (in memory)
parts = []
for t in EXPLORE_TABLES:
    dfd = nlp_detail_frames.get(f'BROAD_{t}')
    if dfd is None or len(dfd) == 0:
        continue
    x = dfd.copy()
    x['SRC_TABLE'] = t
    parts.append(x)
df_broad_all_detail = (
    pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
)
print('All Step A NLP detail rows:', len(df_broad_all_detail))
if len(df_broad_all_detail):
    print('\n=== Overall: keywords across all tables ===')
    display(
        df_broad_all_detail.groupby('NLP_TERM', as_index=False)
        .agg(
            HIT_ROWS=('EXACT_COLUMN_VALUE', 'size'),
            UNIQUE_PATIENTS=('PATIENT_ID', 'nunique'),
            TABLES=('SRC_TABLE', lambda s: ', '.join(sorted(set(s)))),
        )
        .sort_values('UNIQUE_PATIENTS', ascending=False)
    )
    print('\n=== Overall: columns across all tables ===')
    display(
        df_broad_all_detail.groupby(['SRC_TABLE', 'MATCH_COLUMN'], as_index=False)
        .agg(
            HIT_ROWS=('EXACT_COLUMN_VALUE', 'size'),
            UNIQUE_PATIENTS=('PATIENT_ID', 'nunique'),
        )
        .sort_values('HIT_ROWS', ascending=False)
    )

In [ ]:
# Optional: session TEMP only for Step A analysis
session.write_pandas(
    df_broad_summary,
    'ATTR_BROAD_AMYLOID_SUMMARY',
    auto_create_table=True,
    table_type='temporary',
    overwrite=True,
)
session.write_pandas(
    df_broad_all_detail if len(df_broad_all_detail) else pd.DataFrame({'PATIENT_ID': []}),
    'ATTR_BROAD_AMYLOID_NLP_DETAIL',
    auto_create_table=True,
    table_type='temporary',
    overwrite=True,
)
print('TEMPORARY ATTR_BROAD_AMYLOID_SUMMARY / ATTR_BROAD_AMYLOID_NLP_DETAIL written')

## Step B — Confirmed ATTR (ICD + SNOMED + ATTR NLP)

Run **after** Step A review.

**Per column** (same as Step A): COUNT → rows → exact values.
Structured ICD/SNOMED OR'd on the relevant column; codes also as NLP text.

### Coding reference (confirmed)

| Phenotype | SNOMED | ICD-10 |
|-----------|--------|--------|
| ATTRwt | `237877004` | `E85.82` |
| ATTR-CM senile cardiac | `16573007` | `E85.82` |
| hATTR-PN / FAP | `42295001` | `E85.1` |
| Broad amyloidogenic TTR | `442012008` | `E85.81` / `E85.89` |

These codes are also in `ATTR_CODE_NLP` (with and without dots).

### B.0 ATTR NLP lists (words + codes-as-text)

In [ ]:
display(pd.DataFrame({'ATTR_NLP_WORDS': ATTR_NLP_WORDS}))
display(pd.DataFrame({'ATTR_CODE_NLP': ATTR_CODE_NLP}))
print('Step B: per-column scan with ATTR_NLP = words + code-as-NLP')

### B.1 `VISIT_CODE` — per column

In [ ]:
cols_vc = table_columns('VISIT_CODE')
skip = {
    'VISIT_CODE_ID', 'VISIT_ID', 'PATIENT_ID',
    'CODE_SYSTEM', 'FIRM_GLOBAL_ID',
}
cols_vc = [c for c in cols_vc if str(c).upper() not in skip]

_ = run_per_column_scan(
    table='VISIT_CODE',
    term_list=ATTR_CODE_NLP,
    key_prefix='VISIT_CODE',
    include_attr_regex=False,
    columns=cols_vc,
    extra_where_by_col={
        'CODE_VALUE': (
            f"{icd_pred('T.CODE_VALUE')} OR {snomed_pred('T.CODE_VALUE')}"
        ),
    },
)


In [ ]:
_ = run_per_column_scan(
    table='VISIT_CODE',
    term_list=ATTR_CODE_NLP,
    key_prefix='VISIT_CODE',
    include_attr_regex=False,
    columns=['CODE_VALUE'],
    extra_where_by_col={
        'CODE_VALUE': (
            f"{icd_pred('T.CODE_VALUE')} OR {snomed_pred('T.CODE_VALUE')}"
        ),
    },
)

### B.2 `MEDICAL_HISTORY` — per column

In [ ]:
_ = run_per_column_scan(
    table='MEDICAL_HISTORY',
    term_list=ATTR_NLP,
    key_prefix='MEDICAL_HISTORY',
    include_attr_regex=True,
    extra_where_by_col={
        'SNOMED': snomed_pred('T.SNOMED'),
        'SECONDARY_SNOMED': snomed_pred('T.SECONDARY_SNOMED'),
        'VALUE': icd_pred('T.VALUE'),
        'OTHER_VALUE': icd_pred('T.OTHER_VALUE'),
    },
)

### B.3 `SURGICAL_HISTORY` — per column

In [ ]:
_ = run_per_column_scan(
    table='SURGICAL_HISTORY',
    term_list=ATTR_NLP,
    key_prefix='SURGICAL_HISTORY',
    include_attr_regex=True,
    extra_where_by_col={
        'SNOMED': snomed_pred('T.SNOMED'),
        'SECONDARY_SNOMED': snomed_pred('T.SECONDARY_SNOMED'),
    },
)

### B.4 `FAMILY_HISTORY` — per column (FHx ≠ patient confirmed)

In [ ]:
_ = run_per_column_scan(
    table='FAMILY_HISTORY',
    term_list=ATTR_NLP,
    key_prefix='FAMILY_HISTORY',
    include_attr_regex=True,
    extra_where_by_col={'SNOMED': snomed_pred('T.SNOMED')},
)

### B.5 `SPECIALTY_FAMILY_HISTORY` — per column (FHx caveat)

In [ ]:
_ = run_per_column_scan(
    table='SPECIALTY_FAMILY_HISTORY',
    term_list=ATTR_NLP,
    key_prefix='SPECIALTY_FAMILY_HISTORY',
    include_attr_regex=True,
    extra_where_by_col={
        'SNOMED': snomed_pred('T.SNOMED'),
        'SECONDARY_SNOMED': snomed_pred('T.SECONDARY_SNOMED'),
    },
)

### B.6 `SPECIALTY_HISTORY` — per column

In [ ]:
_ = run_per_column_scan(
    table='SPECIALTY_HISTORY',
    term_list=ATTR_NLP,
    key_prefix='SPECIALTY_HISTORY',
    include_attr_regex=True,
    extra_where_by_col={
        'SNOMED': snomed_pred('T.SNOMED'),
        'SECONDARY_SNOMED': snomed_pred('T.SECONDARY_SNOMED'),
    },
)

### B.7 `SOCIAL_HISTORY` — per column

In [ ]:
_ = run_per_column_scan(
    table='SOCIAL_HISTORY',
    term_list=ATTR_NLP,
    key_prefix='SOCIAL_HISTORY',
    include_attr_regex=True,
)

### B.8 `LAB_RESULT` — per column

In [ ]:
_ = run_per_column_scan(
    table='LAB_RESULT',
    term_list=ATTR_NLP,
    key_prefix='LAB_RESULT',
    include_attr_regex=True,
)

### B.9 `VISIT` — per column

In [ ]:
_ = run_per_column_scan(
    table='VISIT',
    term_list=ATTR_NLP,
    key_prefix='VISIT',
    include_attr_regex=True,
)

### B.10 Optional `I43` companion (not confirmation alone)

In [ ]:
confirmed_vc_ids = table_hits.get('VISIT_CODE', set())
if confirmed_vc_ids:
    id_list = ', '.join([f"'{p}'" for p in sorted(confirmed_vc_ids)])
    df_i43 = session.sql(f"""
        SELECT VC.*
        FROM VISIT_CODE VC
        WHERE TRIM(VC.PATIENT_ID) IN ({id_list})
          AND STARTSWITH(UPPER(REPLACE(COALESCE(TO_VARCHAR(VC.CODE_VALUE),''),'.','')), 'I43')
        ORDER BY VC.PATIENT_ID
    """).to_pandas()
else:
    df_i43 = pd.DataFrame()
print('I43 companion rows:', len(df_i43))
display(df_i43)

## Step B union — confirmed ATTR PATIENT_IDs

`INCLUDE_FHX = False` by default. For **why**, use exact-column detail above.

In [ ]:
INCLUDE_FHX = False
STRICT_TABLES = [
    'VISIT_CODE', 'MEDICAL_HISTORY', 'SURGICAL_HISTORY',
    'SPECIALTY_HISTORY', 'SOCIAL_HISTORY', 'LAB_RESULT', 'VISIT',
]
FHX_TABLES = ['FAMILY_HISTORY', 'SPECIALTY_FAMILY_HISTORY']
use_tables = STRICT_TABLES + (FHX_TABLES if INCLUDE_FHX else [])

summary_rows = []
for t in STRICT_TABLES + FHX_TABLES:
    summary_rows.append({
        'TABLE': t,
        'ROLE': 'FHX_REVIEW' if t in FHX_TABLES else 'PATIENT_CONFIRMED',
        'IN_UNION': t in use_tables,
        'MATCH_ROWS': len(table_frames.get(t, pd.DataFrame())),
        'UNIQUE_PATIENTS': len(table_hits.get(t, set())),
    })
df_table_summary = pd.DataFrame(summary_rows)
display(df_table_summary)

all_ids = set()
for t in use_tables:
    all_ids |= table_hits.get(t, set())

hits = []
for pid in sorted(all_ids):
    srcs = [t for t in use_tables if pid in table_hits.get(t, set())]
    fhx = [t for t in FHX_TABLES if pid in table_hits.get(t, set())]
    hits.append({
        'PATIENT_ID': pid,
        'SOURCE_TABLES': ','.join(srcs),
        'N_SOURCE_TABLES': len(srcs),
        'ALSO_FHX_TABLES': ','.join(fhx) if fhx else '',
    })
df_confirmed = pd.DataFrame(hits)
print(f'CONFIRMED ATTR unique patients: {len(df_confirmed)}')
display(df_confirmed)

## Drill-down one PATIENT_ID

In [ ]:
REVIEW_PATIENT_ID = '1895078-pod43'  # e.g. '12345678'

if REVIEW_PATIENT_ID:
    pid = str(REVIEW_PATIENT_ID).strip()
    print('=== Broad (Step A) exact hits ===')
    for t in EXPLORE_TABLES:
        dfd = nlp_detail_frames.get(f'BROAD_{t}')
        if dfd is None or len(dfd) == 0:
            continue
        sub = dfd[dfd['PATIENT_ID'].astype(str) == pid]
        if len(sub):
            print(f'BROAD_{t}: {len(sub)}')
            display(sub)
    print('=== Confirmed ATTR (Step B) exact hits ===')
    for t in STRICT_TABLES + FHX_TABLES:
        dfd = nlp_detail_frames.get(t)
        if dfd is None or len(dfd) == 0:
            continue
        sub = dfd[dfd['PATIENT_ID'].astype(str) == pid]
        if len(sub):
            print(f'{t}: {len(sub)}')
            display(sub)
else:
    print('Set REVIEW_PATIENT_ID to drill down.')

In [ ]:
REVIEW_PATIENT_ID = '1958037-pod43'


if REVIEW_PATIENT_ID:
    pid = str(REVIEW_PATIENT_ID).strip()
    print('=== Broad (Step A) exact hits ===')
    for t in EXPLORE_TABLES:
        dfd = nlp_detail_frames.get(f'BROAD_{t}')
        if dfd is None or len(dfd) == 0:
            continue
        sub = dfd[dfd['PATIENT_ID'].astype(str) == pid]
        if len(sub):
            print(f'BROAD_{t}: {len(sub)}')
            display(sub)
    print('=== Confirmed ATTR (Step B) exact hits ===')
    for t in STRICT_TABLES + FHX_TABLES:
        dfd = nlp_detail_frames.get(t)
        if dfd is None or len(dfd) == 0:
            continue
        sub = dfd[dfd['PATIENT_ID'].astype(str) == pid]
        if len(sub):
            print(f'{t}: {len(sub)}')
            display(sub)
else:
    print('Set REVIEW_PATIENT_ID to drill down.')

## Optional — session TEMPORARY only (Step B)

No permanent warehouse writes.

In [ ]:
session.write_pandas(
    df_confirmed if len(df_confirmed) else pd.DataFrame({'PATIENT_ID': []}),
    'ATTR_CONFIRMED_PATIENTS',
    auto_create_table=True,
    table_type='temporary',
    overwrite=True,
)
session.write_pandas(
    df_table_summary,
    'ATTR_CONFIRMED_TABLE_SUMMARY',
    auto_create_table=True,
    table_type='temporary',
    overwrite=True,
)

detail_parts = []
for t in STRICT_TABLES + FHX_TABLES:
    dfd = nlp_detail_frames.get(t)
    if dfd is None or len(dfd) == 0:
        continue
    x = dfd.copy()
    x['SRC_TABLE'] = t
    detail_parts.append(x)
df_all_nlp_detail = (
    pd.concat(detail_parts, ignore_index=True, sort=False)
    if detail_parts else pd.DataFrame()
)
session.write_pandas(
    df_all_nlp_detail if len(df_all_nlp_detail) else pd.DataFrame({'PATIENT_ID': []}),
    'ATTR_CONFIRMED_NLP_DETAIL',
    auto_create_table=True,
    table_type='temporary',
    overwrite=True,
)
print('TEMPORARY Step B tables written:', len(df_confirmed), 'patients')

In [ ]:
%%sql -r dataframe_1
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.FAMILY_HISTORY
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);

In [ ]:
%%sql -r dataframe_2
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.LAB_RESULT
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);

In [ ]:
%%sql -r dataframe_3
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.MEDICAL_HISTORY
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);

In [ ]:
%%sql -r dataframe_4
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.PATIENT
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);

In [ ]:
%%sql -r dataframe_5
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.SOCIAL_HISTORY
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);

In [ ]:
%%sql -r dataframe_6
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.SPECIALTY_FAMILY_HISTORY
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);

In [ ]:
%%sql -r dataframe_7
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.SPECIALTY_HISTORY
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);

In [ ]:
%%sql -r dataframe_8
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.SURGICAL_HISTORY
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);

In [ ]:
%%sql -r dataframe_9
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.VISIT
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);

In [ ]:
%%sql -r dataframe_10
SELECT *
FROM QUALDERM_MODMED_NEW.PUBLIC.VISIT_CODE
WHERE TRIM(PATIENT_ID) IN (
    '1895078-pod43','1958037-pod43','27372327-pod43','30021695-pod43','32356078-pod43',
    '33377016-pod43','33399733-pod43','34182706-pod43','34381916-pod43','34523441-pod43','7520063-pod43'
);